In [2]:
import os
import sys

sys.path.append("..")  # Adds higher directory to python modules path.
from src import RASPRoutines
import polars as pl
import pandas as pd
import numpy as np
from src import IOFunctions
from src import AnalysisFunctions

RASP = RASPRoutines.RASP_Routines()
IO = IOFunctions.IO_Functions()
A_F = AnalysisFunctions.Analysis_Functions()

import warnings
from copy import copy

warnings.filterwarnings("ignore")

from src import HelperFunctions
H_F = HelperFunctions.Helper_Functions()


In [3]:
# MAKE SURE THESE POINT TO THE CORRECT FOLDERS
overall_folder = r"/scratch/sycamore-asap/ASAP_Imaging_Data/Jonathan/Analysis/20240913_Isoform_syn211_second_run/data_analysis"

In [4]:
protein_string = "C0"
cell_string = "C1"

imtype = ".tif"
cell_size_threshold = np.arange(50, 5000, 50)
lo_size_threshold = np.arange(0, 200, 10)
percentiles = np.arange(0, 100, 5)

In [ ]:
analysis_file = os.path.join(overall_folder, "spot_analysis.csv")
data = pl.read_csv(analysis_file)
if "incell" in data:
    data = data.drop("incell")
data_filtered, q1, q2, IQR = A_F.reject_outliers(
    data, k=4
)
thresholds = np.percentile(
    data_filtered["sum_intensity_in_photons"].to_numpy(),
    percentiles,
)
for cell_size in cell_size_threshold:
    for t, threshold in enumerate(thresholds):
        percentile = percentiles[t]
        end_str = "brightness_threshold"+"_percentile_"+str(int(percentile)).zfill(4)+"_k_4_outliersremoved"
        cell_punctum_analysis = (
            RASP.count_puncta_in_individual_cells_threshold_3D(
                copy(data_filtered),
                analysis_file,
                threshold_lower=threshold,
                cell_string=cell_string,
                protein_string=protein_string,
                lower_cell_size_threshold=cell_size,
                upper_cell_size_threshold=np.inf,
                imtype=".tif",
                blur_degree=1,
                replace_files=True,
                end_string=end_str,
                z_project_first=[True, True],
                dims=2,
                sigma1=2.0,
                sigma2=40.0,
                    spacing=(0.11, 0.11),
                )
            )
        del cell_punctum_analysis

In [5]:
folder_analysis = '/scratch/sycamore-asap/ASAP_Imaging_Data/Jonathan/Analysis/20240913_Isoform_syn211_second_run/data_analysis/New_Cell_Analysis/'
csvs = H_F.file_search(folder_analysis, '.csv', '')

In [6]:
import matplotlib.pyplot as plt
from src import PlottingFunctions
Plotter = PlottingFunctions.Plotter()

In [7]:
PDs = ['S3', 'S5', 'S6', 'S7']
HCs = ['S10', 'S12', 'S13', 'S14']

In [14]:
for file in csvs:
    data = pd.read_csv(file)
    pattern_pds = '|'.join(PDs)
    pattern_hcs = '|'.join(HCs)
    
    # Method 1A: Create boolean mask
    mask_pds = data['image_filename'].str.contains(pattern_pds)
    df_pds = data[mask_pds]
    mask_hcs = data['image_filename'].str.contains(pattern_hcs)
    df_hcs = data[mask_hcs]
    
    pcl_PD = np.array(df_pds['puncta_cell_likelihood'])[~np.isnan(np.array(df_pds['puncta_cell_likelihood']))]
    pcl_HC = np.array(df_hcs['puncta_cell_likelihood'])[~np.isnan(np.array(df_hcs['puncta_cell_likelihood']))]

    density_PD = np.array(df_pds['n_puncta_in_cell'])/np.array(df_pds['area/um2'])
    density_PD = density_PD[~np.isnan(density_PD)]
    density_HC = np.array(df_hcs['n_puncta_in_cell'])/np.array(df_hcs['area/um2'])
    density_HC = density_HC[~np.isnan(density_HC)]

    folder = 'Plots'
    filename = os.path.split(file)[1].split('.csv')[0]+'_PCL.svg'
    fig, axs = Plotter.one_column_plot()
    axs = axs[0]
    bins = np.histogram_bin_edges(np.hstack([pcl_PD, pcl_HC]), 'fd')
    axs = Plotter.histogram_plot(axs, data=pcl_HC, bins=bins, xaxislabel='puncta cell likelihood', alpha=0.5)
    axs = Plotter.histogram_plot(axs, data=pcl_PD, bins=bins, xaxislabel='puncta cell likelihood', histcolor='darkred', alpha=0.5)
    median_HC = np.median(pcl_HC)
    median_PD = np.median(pcl_PD)
    axs.set_yscale('log')
    ymin, ymax = axs.get_ylim()
    axs.vlines(x=median_HC, ymin=ymin, ymax=ymax, ls='--', lw=1, colors='gray', label='HC median')
    axs.vlines(x=median_PD, ymin=ymin, ymax=ymax, ls='--', lw=1, colors='darkred', label='PD median')
    axs.set_ylim([ymin, ymax])
    
    axs.set_xlim([0, np.percentile(np.hstack([pcl_PD, pcl_HC]), 99)])
    axs.legend(loc='best', fontsize=6)
    plt.savefig(os.path.join(folder_analysis, folder, filename), format='svg', dpi=600)
    plt.close()
    filename = os.path.split(file)[1].split('.csv')[0]+'_density.svg'
    fig, axs = Plotter.one_column_plot()
    axs = axs[0]
    bins = np.histogram_bin_edges(np.hstack([density_PD, density_HC]), 'scott')
    axs = Plotter.histogram_plot(axs, data=density_HC, bins=bins, xaxislabel=r'in-cell density/µm$^2$', alpha=0.5)
    axs = Plotter.histogram_plot(axs, data=density_PD, bins=bins, xaxislabel=r'in-cell density/µm$^2$', histcolor='darkred', alpha=0.5)
    median_HC = np.median(density_HC[density_HC > 0])
    median_PD = np.median(density_PD[density_PD > 0])
    axs.set_yscale('log')
    ymin, ymax = axs.get_ylim()
    axs.vlines(x=median_HC, ymin=ymin, ymax=ymax, ls='--', lw=1, colors='gray', label='HC median (excluding 0s)')
    axs.vlines(x=median_PD, ymin=ymin, ymax=ymax, ls='--', lw=1, colors='darkred', label='PD median (excluding 0s)')

    axs.set_ylim([ymin, ymax])
    axs.set_xlim([0, np.percentile(np.hstack([density_HC, density_PD]), 99)])
    axs.legend(loc='best', fontsize=6)
    plt.savefig(os.path.join(folder_analysis, folder, filename), format='svg', dpi=600)
    plt.close()
    

In [12]:
axs.get_ylim()

(0.03429015551088601, 288.21184743682454)

In [ ]:
bins = np.histogram_bin_edges(density, 'sqrt')
plt.hist(density, bins=bins)
plt.show()

In [ ]:
np.percentile(pcl, 100)